# Multi-Armed Bandits: Greedy, ε-Greedy, Explore-First and UCB

**Problems 9. ESE 561, Theory of Artificial Intelligence.**

Run the cells from top to bottom. The cells marked **Question** are for discussion: try to **predict the answer before you run the next cell**, then check.

| ---------notation--------- | this notebook | meaning |
|---|---|---|
| $M$ | `k` | number of arms |
| $R(a)$ | `bandit.true_means[a]` | mean reward of arm $a$ (hidden from the algorithm) |
| $n_t(a)$ | `N[a]` | number of pulls of arm $a$ so far |
| $\hat{R}_t(a)$ | `S[a] / N[a]` | sample mean of arm $a$ |
| $\Delta(a)=R(a^\star)-R(a)$ | `gaps[a]` | gap of arm $a$ |
| $R_T$ | `regret[T-1]` | cumulative regret |

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
plt.rcParams['figure.figsize'] = (7, 4.5)

## 1. The bandit environment

We use the **10-armed testbed** of Sutton & Barto: arm $a$ pays $r\sim\mathcal N(R(a),1)$, Gaussian with unit variance. The means $R(a)$ are drawn once from $\mathcal N(0,1)$ when a bandit is created, so every bandit instance is different. That is why we will **average over many bandits** before drawing conclusions.

The algorithm may only call `pull(a)`. The attributes `true_means`, `best_arm` and `best_mean` are there only for us, to compute the regret.

In [ ]:
class GaussianBandit:
    """k-armed bandit with Gaussian unit-variance rewards (Sutton & Barto testbed)."""
    def __init__(self, k=10, seed=None):
        rng = np.random.default_rng(seed)
        self.k = k
        self.true_means = rng.normal(0, 1, size=k)       # R(a), hidden from the algorithm
        self.best_arm = int(np.argmax(self.true_means))  # a*
        self.best_mean = self.true_means[self.best_arm]  # R(a*)
        self._rng = rng

    def pull(self, a):
        """Pull arm a and return one noisy reward."""
        return self._rng.normal(self.true_means[a], 1.0)


b = GaussianBandit(k=10, seed=1)
print('true means:', b.true_means.round(2))
print('best arm:  ', b.best_arm, '  best mean:', round(b.best_mean, 3))
print('five pulls of arm 1:', np.round([b.pull(1) for _ in range(5)], 2))

**Question 0 (solved).** In this bandit the best arm is 4 ($R=0.91$) and the runner-up is arm 1 ($R=0.82$). The rewards have standard deviation 1. After 5 pulls of each, the sample mean has standard deviation $1/\sqrt5\approx0.45$. Can 5 pulls per arm reliably tell arm 4 from arm 1? How many pulls would you need? Does it matter much if we confuse them?

**Short answer:** no. After 5 pulls each you pick the wrong arm about 45% of the time, and telling them apart reliably takes about 700 pulls of each arm. But confusing them costs very little.

The precise gap is Δ = 0.905 − 0.820 ≈ 0.085 (about 0.09).

### 1. Distribution of the two sample means

Each pull is r ~ N(R(a), 1). The mean of n independent pulls is Gaussian, with variance 1/n:

$$\hat R(4)\sim\mathcal N\!\left(0.905,\tfrac1n\right),\qquad \hat R(1)\sim\mathcal N\!\left(0.820,\tfrac1n\right).$$

For n = 5 the standard deviation is $1/\sqrt{5}$ ≈ 0.45, five times the gap.

### 2. Reduce the comparison to one Gaussian

We pick the wrong arm when $\hat{R}(1) > \hat{R}(4)$. Look at the difference $D = \hat{R}(4) - \hat{R}(1)$. The two means are independent, so their variances add:

$$D\sim\mathcal N\!\left(\Delta,\ \tfrac2n\right).$$

Standardizing gives

$$P(\text{confuse}) = P(D<0)=P\!\left(Z<\frac{0-\Delta}{\sqrt{2/n}}\right)=\Phi\!\left(-\Delta\sqrt{n/2}\right).$$

### 3. Plug in n = 5

$$\Delta\sqrt{n/2}=0.085\cdot\sqrt{2.5}\approx0.134,\qquad \Phi(-0.134)\approx0.45 .$$

A simulation with 200,000 trials gives 0.446. So 5 pulls each is **almost a coin flip** between the two arms.

### 4. How many pulls for a reliable decision?

Require P(confuse) ≤ δ. With z_{1−δ} the standard normal quantile:

$$\Phi\!\left(-\Delta\sqrt{n/2}\right)\le\delta \iff \Delta\sqrt{n/2}\ge z_{1-\delta}\iff n\ \ge\ \frac{2\,z_{1-\delta}^2}{\Delta^2}.$$

| error δ | z | pulls of each arm |
|---|---|---|
| 5% | 1.645 | 2·1.645²/0.085² ≈ **750** |
| 1% | 2.326 | ≈ **1500** |

The probability falls slowly: 0.45 at n = 5, 0.27 at n = 100, 0.09 at n = 500, 0.03 at n = 1000. The rule to remember is **n ∝ 1/Δ²**: halving the gap quadruples the pulls needed.

(And about 2000 for 3 standard deviations margin, z = 3.)

### 5. Does it matter?

Use the regret decomposition. Each pull of arm 1 instead of arm 4 costs only Δ ≈ 0.085:

- **Committing to arm 1 by mistake** for the remaining 950 pulls costs about 0.085 × 950 ≈ 81.
- **Making the decision reliable** costs far more. Doing it with Explore-First means 750 pulls of every arm. That's 7500 pulls, more than T = 1000, and each one costs the average gap $\bar{\Delta}≈ 0.68$ on this bandit.

So it isn't worth it. Arms that are hard to tell apart are, for that very reason, cheap to confuse.

You can make this precise. The expected loss per step from confusing two arms is

$$\Delta\cdot\Phi\!\left(-\Delta\sqrt{n/2}\right).$$

Write u = Δ$\sqrt{n/2}$. The loss becomes $\sqrt{n/2}$ · uΦ(−u), and uΦ(−u) is at most about 0.17, at u ≈ 0.75. So the worst gap is Δ ≈ $1/\sqrt{n}$, and the loss is at most $0.24/\sqrt{n}$ per step whatever the gap:
- smaller gaps are cheap even when confused;
- larger gaps are almost never confused.

This is why the Explore-First analysis only needs Δ(â) ≤ 2β, not â = a*. Where β is the **confidence radius**:  Hoeffding's inequality, for n independent rewards in [0, 1], says

$$P\bigl(|\hat R(a)-R(a)|\ge\beta\bigr)\le 2e^{-2n\beta^2}.$$

Pick the failure probability you can tolerate, δ, and solve 2e^{−2nβ²} = δ:

$$\beta=\sqrt{\frac{\log(2/\delta)}{2n}} .$$

The notes choose δ = 2/T⁴, which gives the formula on the slide:

$$\beta=\sqrt{\frac{2\log T}{n}} .$$

So [R̂(a) − β, R̂(a) + β] is a confidence interval for R(a), with failure probability 2/T⁴ for each arm.

### 6. Why Explore-First with m = 5 picked arm 0, not arm 1

The pairwise calculation understates the risk. With 10 arms, the best arm has to beat **all nine** noisy means at once. Arm 7 (R = 0.58, gap 0.32), for example, beats arm 4 with probability Φ(−0.32·$\sqrt{2.5}$) ≈ 0.31 on its own. In the notebook run, arm 0's five rewards happened to be high, so it won with a gap of 0.56. That's the expensive kind of mistake, 566 in regret: a larger gap confused through bad luck, which is what the 2β bound guards against.


### **Question 1.** Warm-up: be the algorithm

Set `PLAY = True` and run the cell. You get 20 pulls of a fresh 5-armed bandit; type an arm number each time. At the end you see your regret and the true means. What strategy did you use?

In [ ]:
PLAY = False

def play_yourself(T=20, k=5, seed=None):
    bandit = GaussianBandit(k=k, seed=seed)
    regret = 0.0
    for t in range(1, T + 1):
        a = int(input(f'step {t}/{T}, pick an arm 0..{k-1}: '))
        r = bandit.pull(a)
        regret += bandit.best_mean - bandit.true_means[a]
        print(f'   arm {a} paid {r:+.2f}')
    print(f'\nyour regret: {regret:.2f}')
    print('true means:', bandit.true_means.round(2), ' best arm:', bandit.best_arm)

if PLAY:
    play_yourself()

## 2. Regret

Because we generated the bandit, we know $R(a^\star)$ even though the algorithm does not. The regret after $T$ pulls is
$$R_T=\sum_{t=1}^T\bigl(R(a^\star)-R(a_t)\bigr)=\sum_{t=1}^T\Delta(a_t)=\sum_a\Delta(a)\,n_T(a).$$
The last form, the **regret decomposition**, says that the regret is paid only by pulls of suboptimal arms, each costing its gap. A good algorithm has **sublinear** regret ($R_T/T\to0$); an algorithm that keeps pulling a bad arm a fixed fraction of the time has **linear** regret.

The helper below turns a sequence of chosen arms into the cumulative regret curve.

In [ ]:
def cumulative_regret(bandit, arms):
    """R_t for t = 1..T, from the sequence of arms chosen."""
    return np.cumsum(bandit.best_mean - bandit.true_means[arms])


# check the regret decomposition on a random sequence of arms
b = GaussianBandit(k=10, seed=1)
arms = np.random.default_rng(0).integers(10, size=500)
gaps = b.best_mean - b.true_means
counts = np.bincount(arms, minlength=10)
print('sum over steps of Delta(a_t):', cumulative_regret(b, arms)[-1].round(3))
print('sum over arms of Delta(a) n(a):', (gaps * counts).sum().round(3))

## 3. Greedy and ε-greedy

**Greedy:** pull every arm once, then always pull the arm with the best sample mean. This is the deterministic policy $\pi(s)=\arg\max_a R(a)$ from the MDP lecture with the unknown $R$ replaced by its estimate.

**ε-greedy** *deliberately randomizes* the greedy policy: with probability $1-\varepsilon$ pull the greedy arm, with probability $\varepsilon$ pull a uniformly random arm. With a known model, randomizing can only lose reward. Here it buys **information**: every arm keeps being pulled now and then, so every estimate keeps improving.

`eps` can be a number, or a function of $t$ for a decaying schedule. `eps=0` is greedy.

In [ ]:
def epsilon_greedy(bandit, T, eps=0.1, seed=0):
    """Returns the rewards and the arms chosen. eps: a number, or a function t -> eps_t."""
    rng = np.random.default_rng(seed)        # the agent's own coin flips
    k = bandit.k
    N, S = np.zeros(k), np.zeros(k)          # pull counts and reward sums
    rewards, arms = np.zeros(T), np.zeros(T, dtype=int)
    for t in range(T):
        e = eps(t + 1) if callable(eps) else eps
        if t < k:
            a = t                            # pull every arm once
        elif rng.random() < e:
            a = int(rng.integers(k))         # explore: random arm
        else:
            a = int(np.argmax(S / N))        # exploit: best sample mean
        r = bandit.pull(a)
        N[a] += 1
        S[a] += r
        rewards[t], arms[t] = r, a
    return rewards, arms


b = GaussianBandit(k=10, seed=1)
for eps in [0.0, 0.1]:
    _, arms = epsilon_greedy(GaussianBandit(k=10, seed=1), T=1000, eps=eps)
    print(f'eps = {eps}:  pulls per arm {np.bincount(arms, minlength=10)},  regret {cumulative_regret(b, arms)[-1]:.1f}')

**Question 2.** Predict before running the next cell: on what fraction of random 10-armed bandits does **greedy** end up pulling a wrong arm forever? How does that fraction change with ε = 0.01?

_your answer (guess)_

In [ ]:
def fraction_stuck(eps, n_bandits=200, T=1000):
    stuck = 0
    for i in range(n_bandits):
        b = GaussianBandit(k=10, seed=1000 + i)
        _, arms = epsilon_greedy(b, T, eps=eps, seed=i)
        stuck += np.argmax(np.bincount(arms[-200:], minlength=10)) != b.best_arm   # most pulled arm at the end
    return stuck / n_bandits

for eps in [0.0, 0.01, 0.1]:
    print(f'eps = {eps:<5}: fraction of bandits where the most pulled arm in the last 200 steps is not the best: {fraction_stuck(eps):.2f}')

**Question 3.** Why can greedy get stuck forever, while ε-greedy cannot? (Think about what happens to $\hat R_t(a^\star)$ if the first reward of the best arm is unlucky.) Why does ε-greedy with $\varepsilon=0.1$ still pull a wrong arm about 10% of the time, even after it has found the best one?

_your answer_

## 4. Explore-First

Pull every arm $m$ times (round robin), then commit to the arm with the best sample mean for the remaining $T-km$ steps. The two phases never overlap.

In [ ]:
def explore_first(bandit, T, m=10):
    """Returns the rewards and the arms chosen."""
    k = bandit.k
    N, S = np.zeros(k), np.zeros(k)
    rewards, arms = np.zeros(T), np.zeros(T, dtype=int)
    for t in range(T):
        if t < k * m:
            a = t % k                         # exploration phase: round robin
        else:
            if t == k * m:
                best = int(np.argmax(S / N))  # commit once, to the best sample mean
            a = best
        r = bandit.pull(a)
        N[a] += 1
        S[a] += r
        rewards[t], arms[t] = r, a
    return rewards, arms


b = GaussianBandit(k=10, seed=1)
for m in [5, 10, 20, 50]:
    _, arms = explore_first(GaussianBandit(k=10, seed=1), T=1000, m=m)
    print(f'm = {m:2d}: committed to arm {arms[-1]} (best is {b.best_arm}),  regret {cumulative_regret(b, arms)[-1]:6.1f}')

**Question 4.** On this bandit, why is the regret for $m=5$ so large, and why does it grow again for $m=20$ and $m=50$? (Hint: during exploration every arm is pulled equally often.)

_Your answer_

## 5. Upper Confidence Bound (UCB)

After pulling every arm once, pick
$$a_t=\arg\max_a\;\underbrace{\hat R_t(a)}_{\text{exploitation}}+\underbrace{c\sqrt{\frac{\log t}{n_t(a)}}}_{\text{exploration bonus}} .$$
$c=\sqrt2$ is the Hoeffding bonus from the notes (with $\log t$ instead of $\log T$, so the algorithm does not need to know $T$). There is no separate exploration phase and no randomness: the bonus of an arm shrinks every time it is pulled, so UCB explores the arms it is **uncertain** about, and stops exploring an arm once its upper bound falls below the best arm's mean.

The function also returns the two terms for the chosen arm, so we can watch them in Section 8.

In [ ]:
def ucb(bandit, T, c=np.sqrt(2)):
    """Returns rewards, arms, and the exploitation / exploration terms of the chosen arm (NaN for the first k pulls)."""
    k = bandit.k
    N, S = np.zeros(k), np.zeros(k)
    rewards, arms = np.zeros(T), np.zeros(T, dtype=int)
    exploit, explore = np.full(T, np.nan), np.full(T, np.nan)
    for t in range(T):
        if t < k:
            a = t                                      # pull every arm once
        else:
            means = S / N
            bonus = c * np.sqrt(np.log(t + 1) / N)
            a = int(np.argmax(means + bonus))
            exploit[t], explore[t] = means[a], bonus[a]
        r = bandit.pull(a)
        N[a] += 1
        S[a] += r
        rewards[t], arms[t] = r, a
    return rewards, arms, exploit, explore


b = GaussianBandit(k=10, seed=1)
_, arms, _, _ = ucb(GaussianBandit(k=10, seed=1), T=1000)
print('pulls per arm:', np.bincount(arms, minlength=10), '  regret', cumulative_regret(b, arms)[-1].round(1))

**Question 5.** What does UCB do when $c=0$? And when $c$ is huge?

_Your answer_

## 5b. Thompson sampling

A Bayesian way to explore. Keep a **posterior** distribution over each mean $R(a)$; at every step **sample** one plausible value $\tilde R(a)$ from each posterior and pull the arm with the largest sample:
$$\tilde R(a)\sim p\bigl(R(a)\,\big|\,h_{t-1}\bigr),\qquad a_t=\arg\max_a\tilde R(a).$$
Each arm is pulled with exactly the probability that it is the best one given the data ("probability matching"). Like ε-greedy it randomizes, but an arm that is clearly bad has a posterior far below the others and is almost never sampled on top.

For our testbed the computation is three lines. With the prior $R(a)\sim\mathcal N(0,\sigma_0^2)$ and unit-variance rewards, the posterior after $n_t(a)$ pulls with reward sum $S_t(a)$ is Gaussian with
$$\text{variance } v_t(a)=\frac{1}{1/\sigma_0^2+n_t(a)},\qquad \text{mean } v_t(a)\,S_t(a).$$
The default $\sigma_0=1$ is exactly how the testbed draws its means.

In [ ]:
def thompson(bandit, T, prior_sd=1.0, seed=0):
    # Gaussian Thompson sampling. Returns the rewards and the arms chosen.
    rng = np.random.default_rng(seed)
    k = bandit.k
    N, S = np.zeros(k), np.zeros(k)
    rewards, arms = np.zeros(T), np.zeros(T, dtype=int)
    for t in range(T):
        post_var = 1.0 / (1.0 / prior_sd**2 + N)      # posterior variance of each R(a)
        post_mean = post_var * S                       # posterior mean of each R(a)
        sample = rng.normal(post_mean, np.sqrt(post_var))
        a = int(np.argmax(sample))                     # greedy with respect to the sample
        r = bandit.pull(a)
        N[a] += 1
        S[a] += r
        rewards[t], arms[t] = r, a
    return rewards, arms


b = GaussianBandit(k=10, seed=1)
_, arms = thompson(GaussianBandit(k=10, seed=1), T=1000)
print('pulls per arm:', np.bincount(arms, minlength=10), '  regret', cumulative_regret(b, arms)[-1].round(1))

**Question 6.** Thompson sampling has no exploration constant like $c$ or $\varepsilon$. What plays its role? Run the next cell (about 10 seconds) to see the regret at $T=1000$ and different prior $\sigma$.

In [ ]:
for sd in [0.1, 0.3, 1, 3, 10]:
    total = 0.0
    for run in range(100):
        b = GaussianBandit(10, seed=1000 + run)
        _, arms = thompson(b, 1000, prior_sd=sd, seed=run)
        total += cumulative_regret(b, arms)[-1]
    print(f'prior sd = {sd:<4}:  regret at T = 1000: {total / 100:6.1f}')

_Your answer_

## 6. Comparison: cumulative regret

Every algorithm runs on the **same** 200 bandits (same means and same reward noise), and we average the cumulative regret over them. This takes about 20 seconds.

In [ ]:
def run_experiment(algorithms, k=10, T=2000, n_runs=200, seed0=1000):
    """algorithms: dict name -> function(bandit, T, run) returning the arms chosen.
    Returns dict name -> (n_runs, T) array of cumulative regret."""
    regret = {name: np.zeros((n_runs, T)) for name in algorithms}
    for run in range(n_runs):
        for name, algo in algorithms.items():
            bandit = GaussianBandit(k, seed=seed0 + run)
            regret[name][run] = cumulative_regret(bandit, algo(bandit, T, run))
    return regret


algorithms = {
    'greedy':              lambda b, T, run: epsilon_greedy(b, T, eps=0.0, seed=run)[1],
    'ε-greedy, ε = 0.1':   lambda b, T, run: epsilon_greedy(b, T, eps=0.1, seed=run)[1],
    'ε-greedy, ε = 0.01':  lambda b, T, run: epsilon_greedy(b, T, eps=0.01, seed=run)[1],
    'Explore-First, m = 10': lambda b, T, run: explore_first(b, T, m=10)[1],
    'UCB, c = √2':         lambda b, T, run: ucb(b, T, c=np.sqrt(2))[1],
    'Thompson sampling':   lambda b, T, run: thompson(b, T, seed=run)[1],
}
T = 2000
regret = run_experiment(algorithms, T=T)
for name, R in regret.items():
    print(f'{name:22s}  R_T = {R[:, -1].mean():6.1f}')

In [ ]:
t_axis = np.arange(1, T + 1)
for name, R in regret.items():
    plt.plot(t_axis, R.mean(axis=0), label=name)
plt.xlabel('time step t'); plt.ylabel('average cumulative regret')
plt.title('Cumulative regret, averaged over 200 bandits')
plt.legend(); plt.grid(alpha=.3); plt.show()

**Question 7.** Which curves become straight lines (linear regret) and which keep bending (sublinear)? Explain each shape from the algorithm: where does greedy's slope come from? Explore-First's two regimes? Which algorithm is best at $t=200$, and which at $t=2000$?

_Your answer_

**Note** For ε-greedy with $\varepsilon=0.1$ the notes predict a slope of about $\varepsilon\bar\Delta$ per step once the best arm is known, where $\bar\Delta=\frac1M\sum_a\Delta(a)$. Compute $\bar\Delta$ averaged over our 200 bandits and compare with the measured slope.

In [ ]:
mean_gap = np.mean([GaussianBandit(10, seed=1000 + run).best_mean - GaussianBandit(10, seed=1000 + run).true_means.mean()
                    for run in range(200)])
R = regret['ε-greedy, ε = 0.1'].mean(axis=0)
slope = (R[-1] - R[T // 2]) / (T - 1 - T // 2)
print(f'average gap = {mean_gap:.3f},  eps * average gap = {0.1 * mean_gap:.3f},  measured slope = {slope:.3f}')

$\bar\Delta\approx1.53$, so the prediction is $\varepsilon\bar\Delta\approx0.153$ per step, and the measured slope is about $0.16$. The prediction is a lower bound: it counts only the cost of the random pulls, while the greedy part also still picks a wrong (usually nearly tied) arm in some runs.

## 7. Who pays the regret? UCB pulls a bad arm about $\log T/\Delta^2$ times

By the regret decomposition, what matters is how often each suboptimal arm is pulled. The notes show that UCB pulls an arm with gap $\Delta$ at most about $8\log T/\Delta^2$ times. Below, every dot is one suboptimal arm of one bandit.

In [ ]:
T7 = 2000
gaps_all, counts_all = [], []
for run in range(100):
    b = GaussianBandit(10, seed=1000 + run)
    _, arms, _, _ = ucb(b, T7)
    g = b.best_mean - b.true_means
    n = np.bincount(arms, minlength=10)
    mask = g > 0
    gaps_all += list(g[mask]); counts_all += list(n[mask])

gaps_all, counts_all = np.array(gaps_all), np.array(counts_all)
grid = np.logspace(-4, 0.7, 200)
plt.loglog(gaps_all, counts_all, '.', alpha=.4, label='suboptimal arms (100 bandits)')
plt.loglog(grid, np.minimum(8 * np.log(T7) / grid**2, T7), 'k--', label='8 log T / Δ²  (capped at T)')
plt.xlabel('gap Δ(a)'); plt.ylabel('number of pulls n_T(a)')
plt.title(f'UCB, T = {T7}'); plt.legend(); plt.grid(alpha=.3, which='both'); plt.show()

**Question 9 (Solved).** Which arms does UCB pull a lot, and which almost never? Why are arms with a tiny gap pulled many times but still cheap in terms of regret? Compare with ε-greedy, which pulls every arm about $\varepsilon T/M$ times regardless of its gap.

**Answer.** Arms with a large gap (bottom right) are pulled only once or a few times: their upper bound drops below $R(a^\star)$ almost immediately. As the gap shrinks the number of pulls grows like $1/\Delta^2$, following the $8\log T/\Delta^2$ line (most dots are below it, because the constant is a worst case). Arms with a tiny gap are pulled hundreds of times, but each pull costs only $\Delta$, so their total cost $\Delta\,n_T(a)\lesssim8\log T/\Delta$ (and at most $\Delta T$) is small. ε-greedy pulls every arm about $\varepsilon T/M$ times regardless of its gap, so the terrible arms (gap 2–3) cost the most, which is exactly where UCB spends almost nothing.

## 8. Inside UCB: exploitation vs. exploration term

For the arm UCB actually picks, we record the two terms of its index and average them over 200 bandits.

In [ ]:
import warnings
warnings.filterwarnings('ignore', message='Mean of empty slice')   # the first k steps have no UCB terms
exploit_all, explore_all = [], []
for run in range(200):
    _, _, ex, bo = ucb(GaussianBandit(10, seed=1000 + run), T)
    exploit_all.append(ex); explore_all.append(bo)

plt.plot(t_axis, np.nanmean(exploit_all, axis=0), label='exploitation term  (sample mean of the chosen arm)')
plt.plot(t_axis, np.nanmean(explore_all, axis=0), label='exploration term  (bonus of the chosen arm)')
best = np.mean([GaussianBandit(10, seed=1000 + run).best_mean for run in range(200)])
plt.axhline(best, color='gray', ls=':', label='average R(a*)')
plt.xlabel('time step t'); plt.ylabel('term value'); plt.title('UCB: the two terms of the chosen arm')
plt.legend(); plt.grid(alpha=.3); plt.show()

**Question 10.** Describe the two curves. Where do they cross (e.g., number if steps)? Nothing in the code says "now stop exploring": what makes UCB shift from exploring to exploiting?

_Your answer_

## 9. Tuning: every method has a knob

Final regret at $T=1000$ (100 bandits) as a function of $m$ (Explore-First), $\varepsilon$ (ε-greedy) and $c$ (UCB). About 15 seconds.

In [ ]:
def final_regret(algo, T=1000, n_runs=100):
    return run_experiment({'x': algo}, T=T, n_runs=n_runs)['x'][:, -1].mean()

ms = [1, 2, 5, 10, 20, 50]
epss = [0, 0.01, 0.05, 0.1, 0.2, 0.5]
cs = [0, 0.25, 0.5, 1, np.sqrt(2), 2, 4]
reg_m = [final_regret(lambda b, T, run, m=m: explore_first(b, T, m)[1]) for m in ms]
reg_e = [final_regret(lambda b, T, run, e=e: epsilon_greedy(b, T, e, seed=run)[1]) for e in epss]
reg_c = [final_regret(lambda b, T, run, c=c: ucb(b, T, c)[1]) for c in cs]

fig, axes = plt.subplots(1, 3, figsize=(15, 4), sharey=True)
for ax, x, y, name in [(axes[0], ms, reg_m, 'Explore-First: m'), (axes[1], epss, reg_e, 'ε-greedy: ε'),
                       (axes[2], cs, reg_c, 'UCB: c')]:
    ax.plot(range(len(x)), y, 'o-')
    ax.set_xticks(range(len(x))); ax.set_xticklabels([f'{v:.2g}' for v in x])
    ax.set_xlabel(name); ax.grid(alpha=.3)
axes[0].set_ylabel('regret at T = 1000')
plt.tight_layout(); plt.show()

**Question 11 (Solved).** All three curves are U-shaped. For each method, explain what goes wrong at the left end (too little exploration) and at the right end (too much). Which point of the UCB curve is greedy? Is the theoretical $c=\sqrt2$ the best value here? Does the best $m$ for Explore-First depend on $T$?

**Answer.** Explore-First: small $m$ commits to wrong arms (regret paid until $T$), large $m$ wastes $km$ pulls at the average gap; best $m\approx5$ at $T=1000$. The optimal $n$ grows like $T^{2/3}$, so the best $m$ depends on $T$, which is the main practical weakness of the method. ε-greedy: $\varepsilon=0$ is greedy (stuck runs), large $\varepsilon$ costs $\varepsilon\bar\Delta T$; best around $0.01$–$0.05$. UCB: $c=0$ is greedy (the left end, regret about 154), $c$ between $0.5$ and $1$ is best (about 60), $\sqrt2$ gives about 90, and $c=4$ explores far too much (about 380). The theoretical $c=\sqrt2$ comes from a worst-case bound (Hoeffding plus union bounds), so it is conservative; but it is a safe choice, whereas a too-small $c$ brings back the stuck-forever failure.